[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://github.com/codespaces)


<!-- gen:values.yml#weeks.05.notebook-title|Debrief -->
# Week 5 Debrief: Advanced pandas
<!-- /gen -->

*Worked answers for [Quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest) 05: The Pivot Pitch, and this week's [AI failure gallery](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-failure-gallery). Read it after you have attempted the quest.*

As always, use this document alongside your own **quest**, not instead of it (obviously). For each stage, compare your answer with the worked one, and note the differences in your [QuestLog](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-questlog).

These are example answers, **not the only answers**. An answer of your own that does what it's supposed to do (and that you can explain) is always best.

Run the data-setup cell below first.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
import os
import numpy as np
import pandas as pd

DATA_DIR = "assets/data/week05"
os.makedirs(DATA_DIR, exist_ok=True)
rng = np.random.default_rng(5813)  # module seed: Do not change

# Marlow & Finch: 12 home-and-gifts stores in 3 regions, daily takings,
# 18 months (2025-01-01 to 2026-06-30), with a promotion flag per store-day.
REGIONS = {
    "North":    ["Harrogate", "Ilkley", "Skipton", "York"],
    "Midlands": ["Leamington", "Ludlow", "Stamford", "Buxton"],
    "South":    ["Marlow", "Henley", "Guildford", "Lewes"],
}
lift = {"North": 1.15, "Midlands": 1.00, "South": 1.12}

dates = pd.date_range("2025-01-01", "2026-06-30", freq="D")
frames, store_id = [], 100
for region, towns in REGIONS.items():
    for town in towns:
        store_id += 1
        base = rng.uniform(2600, 4200)
        weekend = dates.dayofweek >= 5
        promo = rng.random(len(dates)) < np.where(weekend, 0.45, 0.04)
        season = 1 + 0.15 * np.cos(2 * np.pi * (dates.dayofyear - 355) / 365)
        sales = (base * season
                 * np.where(weekend, 1.28, 1.0)
                 * np.where(promo, lift[region], 1.0)
                 * rng.normal(1, 0.06, len(dates)))
        frames.append(pd.DataFrame({"date": dates, "store": f"MF-{store_id}",
                                    "town": town, "region": region,
                                    "promo": promo, "sales": sales.round(2)}))

daily_data = pd.concat(frames, ignore_index=True)
daily_data.to_csv(f"{DATA_DIR}/mf_daily_sales.csv", index=False)

# The raw till export: On promo days the till writes TWO rows per store.
# Promo-priced lines and standard lines land on separate streams.
promo_rows = daily_data[daily_data["promo"]]
share = rng.uniform(0.35, 0.55, len(promo_rows))
promo_lines = promo_rows.assign(stream="promo_lines",
                                sales=(promo_rows["sales"] * share).round(2))
standard_lines = promo_rows.assign(stream="standard_lines",
                                   sales=(promo_rows["sales"] - promo_lines["sales"]).round(2))
till_feed = (pd.concat([daily_data[~daily_data["promo"]].assign(stream="standard_lines"),
                        promo_lines, standard_lines], ignore_index=True)
             .sample(frac=1, random_state=5813)
             .reset_index(drop=True))
till_feed.to_csv(f"{DATA_DIR}/mf_till_feed.csv", index=False)

print(f"Created {DATA_DIR}/mf_daily_sales.csv ({len(daily_data)} rows, one per store per day)")
print(f"Created {DATA_DIR}/mf_till_feed.csv ({len(till_feed)} rows, raw till streams)")

The cell below reloads the working file and rebuilds the `day_type` column, as in the quest.

In [ ]:
# Load the working file, as in the quest
daily = pd.read_csv(f"{DATA_DIR}/mf_daily_sales.csv", parse_dates=["date"])
daily["day_type"] = np.where(daily["date"].dt.dayofweek >= 5,
                             "weekend", "weekday")
print(daily.shape)

## 🔍 First look: The expected output

- **[First look 1](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-first-look):** The table has 6,552 rows and 6 columns. From the CSV, `promo` comes back as `bool` and `sales` as `float64`. The `date` column is `datetime64` because you asked for it with `parse_dates`. Always predict the column types (`dtypes`)
- **First look 2 (the key cell):** Roughly 45% of weekend store-days are promo days, against about 4% of weekdays. So most promo days are weekend days. Any raw comparison of promo days with no-promo days therefore includes the weekend effect
- **First look 3:** The result is a `Series` with a `MultiIndex`, two levels of row labels (region and promo). A board table needs a grid, which `unstack()` or `pivot_table()` makes
- **First look 4:** Promo-day counts are similar across regions (roughly 340-360 store-days per region over 18 months). So the number of promo days a region ran does not explain any difference in effect between regions

## 🥉 Bronze: Worked answer

The shape sentence is the whole specification. Rows are regions, columns are promo and no-promo, cells are average daily sales, and the `aggfunc` is named. The cell below builds the table with one [`pivot_table()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html) call. It then checks one cell by hand.

In [ ]:
bronze = daily.pivot_table(values="sales", index="region",
                           columns="promo", aggfunc="mean").round(0)
print(bronze)

# The hand check: One cell, computed independently
north_promo = daily[(daily["region"] == "North") & daily["promo"]]["sales"].mean()
print(f"\nHand check of the North promo mean: £{north_promo:,.0f} "
      f"(table says £{bronze.loc['North', True]:,.0f})")

The table has 3 rows and 2 columns, with no empty cells (`NaN`). Average sales on promo days are higher than on no-promo days in every region. The hand check matches, so the table is correct. Whether it is evidence is [Silver](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-silver)'s question.

## 🥈 Silver: The judgement

**Why the drafts disagree.** Draft A's promo pool is about 80% weekend days, and its no-promo pool is about 80% weekdays. Weekends at Marlow & Finch trade roughly 28% higher with or without a promotion. So Draft A's "uplift" is mostly the weekend effect. That is why Draft A finds a large uplift even in the Midlands. Draft B keeps the day type fixed. It compares promo days with no-promo days within weekends only, which isolates what the director asked about.

**The verdict the numbers support:** Weekend promotions work in the **North** (roughly +18%, weekend-matched) and the **South** (roughly +10%). They do **nothing detectable in the Midlands** (about +2%, within noise). Draft A's advice to "extend chain-wide" would have sent promotion budget to the one region where promotions achieve nothing.

The cell below builds one defensible version of the refined board table. It adds an uplift column and the number of promo days behind each row.

In [ ]:
weekend_only = daily[daily["day_type"] == "weekend"]

board = weekend_only.pivot_table(values="sales", index="region",
                                 columns="promo", aggfunc="mean").round(0)
board.columns = ["ordinary weekend (£/day)", "promo weekend (£/day)"]
board["uplift (%)"] = ((board["promo weekend (£/day)"]
                        / board["ordinary weekend (£/day)"] - 1) * 100).round(1)
# Named aggregation: The count the board will ask for ("based on how many days?");
# summing the boolean promo column counts the True days
counts = weekend_only.groupby("region").agg(promo_days=("promo", "sum"))
board["promo days"] = counts["promo_days"]
board = board.sort_values("uplift (%)", ascending=False)

print("Weekend promotions: like-for-like comparison, 18 months")
print(board)

**The sentences for the board:**

*"Comparing promotion weekends with ordinary weekends (like for like), promotions lift takings by about 18% in the North and 10% in the South. In the Midlands the lift is about 2%, indistinguishable from noise, across roughly 270 promotion days. We recommend keeping the programme in the North and South, and pausing it in the Midlands while we find out why."*

A good defence:

- Names what else differs: Weekends trade higher anyway
- Reads a separate answer for each region out of the table, not one yes or no
- Ends in a <!-- plain-language: everyday decision -->decision

Draft B has its own limit. Promo weekends may differ from ordinary weekends in other ways, such as bank holidays and December. A store-matched or seasonally adjusted comparison would make the answer more precise. That is [Gold](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-gold) work. If you spotted it, say so in your defence.

## 🥇 Gold: Worked sketches

**Sketch 1 (the Names option): Weekend-matched uplift per store.** The cell below pivots the weekend data by store. With one row per store, the board can see which stores the effect comes from.

In [ ]:
per_store = (weekend_only
             .pivot_table(values="sales", index=["region", "store", "town"],
                          columns="promo", aggfunc="mean"))
per_store["uplift_%"] = ((per_store[True] / per_store[False] - 1) * 100).round(1)
per_store = (per_store[["uplift_%"]]
             .sort_values("uplift_%", ascending=False)
             .round(1))
print(per_store)

Every North store shows an uplift of 13-20%, and every South store 8-14%. Every Midlands store is within three percentage points of zero. The effect is **regional and consistent**, not two strong stores pulling the average up. That strengthens the recommendation to pause the Midlands.

**Sketch 2 (the Trend option): A [28-day rolling mean](https://pandas.pydata.org/docs/user_guide/window.html) per region.** The cell below sums sales per region per day. It then uses `transform()` to keep the rolling window inside each region.

Here `lambda s: s.rolling(28).mean()` is a one-line unnamed function. It turns each region's sales into a 28-day rolling mean (optional background: [Week 2b](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/02b/Introduction.ipynb)).

In [ ]:
import matplotlib.pyplot as plt

region_daily = (daily.groupby(["date", "region"], as_index=False)["sales"].sum()
                .sort_values(["region", "date"]))
region_daily["rolling_28"] = (region_daily.groupby("region")["sales"]
                              .transform(lambda s: s.rolling(28).mean()))

pivoted_trend = region_daily.pivot_table(values="rolling_28",
                                         index="date", columns="region")
ax = pivoted_trend.plot(figsize=(10, 4),
                        title="28-day rolling mean of daily sales by region")
ax.set_ylabel("sales (£/day, 28-day mean)")
plt.show()

The seasonal rise (the December peak) affects every region alike. That is why the matched comparison, not the trend chart, answers the director's question. In the board pack, the chart gives context, not proof.

Whatever its form, a Gold answer ends with a sentence about the <!-- plain-language: everyday decision -->**decision**. For example:

*"The per-store table shows the same Midlands result in every store, so the fix is the programme's design, not one store's execution."*

## 🚩 Red Flags: Worked answer

**The bug.** The grain of a table is what one row stands for. The till feed's grain is not one row per store-day. On promotion days the till writes **two rows** (promo-priced lines and standard lines), each with part of the day's takings. The [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate)'s pivot passed no `aggfunc`, so `pivot_table()` silently used its default, **`"mean"` over rows**. Every promo day entered the average twice, at roughly half its true value. No-promo days entered once, at full value.

**In business terms.** The table compares whole ordinary days with half promotion days, so promotions appear to cut takings by a third. The board would have cancelled a programme that earns double-digit uplift in two regions.

**The fix, and the size of the error.** The cell below sums the till feed to one row per store-day with `groupby()`, then pivots. It prints the gap between the wrong and the fixed uplift. It then checks the fixed table against [Bronze](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bronze).

In [ ]:
till = pd.read_csv(f"{DATA_DIR}/mf_till_feed.csv", parse_dates=["date"])

# The wrong table, as the Associate built it
wrong = till.pivot_table(values="sales", index="region", columns="promo").round(0)
wrong_uplift = ((wrong[True] / wrong[False] - 1) * 100).round(1)

# The fix: Collapse to the correct grain FIRST (one row per store-day), then pivot
day_level = till.groupby(["date", "store", "region", "promo"],
                         as_index=False)["sales"].sum()
fixed = day_level.pivot_table(values="sales", index="region",
                              columns="promo", aggfunc="mean").round(0)
fixed_uplift = ((fixed[True] / fixed[False] - 1) * 100).round(1)

quantified = pd.DataFrame({"wrong uplift (%)": wrong_uplift,
                           "fixed uplift (%)": fixed_uplift,
                           "gap (pp)": (fixed_uplift - wrong_uplift).round(1)})
print(quantified)

# And the fixed till-feed table matches Bronze, as it must:
bronze = daily.pivot_table(values="sales", index="region",
                           columns="promo", aggfunc="mean").round(0)
print(f"\nFixed table matches Bronze: {fixed.equals(bronze)}")

One omitted argument changes the apparent uplift by some **sixty percentage points**. Here is a directing prompt that fixes it: *"The till feed has two rows per store-day on promo days: Promo lines and standard lines. The table's grain must be one row per store-day. Collapse to that grain with a groupby-sum first, then pivot with aggfunc='mean', and show me the row counts before and after so I can verify."*

Build the habit: Before any `pivot_table()`, say the **grain** of the input out loud. State `aggfunc` every time, because the default is a statistical <!-- plain-language: everyday decision -->decision you did not make.

## 🤖 AI failure gallery

### Exhibit A: The `NaN` grid read as zeros

**The plausible output.** Asked for "average weekday promo sales by store and month", the Associate builds an `unstack()` grid. It sees the gaps and "cleans" them with `.fillna(0)`. It then reports that *"several stores recorded £0 promo sales in some months; promotion execution appears inconsistent."* The cell below builds that grid and its cleaned version.

In [ ]:
wd_promo = daily[(daily["day_type"] == "weekday") & daily["promo"]]
grid = (wd_promo
        .groupby(["store", wd_promo["date"].dt.to_period("M")])["sales"]
        .mean()
        .unstack())

print(f"Empty cells in the grid: {int(grid.isna().sum().sum())} "
      f"of {grid.size}")
print("\nThe Associate's 'cleaned' version (first 3 stores, first 6 months):")
print(grid.fillna(0).round(0).iloc[:3, :6])

**Why it is wrong in business terms.** Weekday promotions are rare (about 4% of weekdays), so many store-months contain no weekday promo at all. `NaN` means "nothing to observe", not "promo took £0". Filling with zero invents disastrous promo days that never happened.

**The catch.** Before you accept any `fillna(0)`, ask whether that store could have taken £0 in that month. Check counts alongside means (`.agg(["mean", "count"])` or `.size()`). A value of 0 with a count of 0 behind it is made up. Only a count of events can be zero.

### Exhibit B: The chained assignment that never happened

**The plausible output.** Asked to recode one store's region ahead of a boundary change, the Associate writes the common two-step form. It filters, then it assigns. The cell below runs it and reads the result back.

In [ ]:
relabelled = daily.copy()

# The Associate's 'quick recode'. It looks right and does NOTHING
relabelled[relabelled["store"] == "MF-101"]["region"] = "North East"

print(relabelled["region"].value_counts())
print("\nMF-101 is still in:",
      relabelled.loc[relabelled['store'] == 'MF-101', 'region'].unique())

**Why it is wrong in business terms.** Since pandas 3.0 ([Copy-on-Write](https://pandas.pydata.org/docs/user_guide/copy_on_write.html)), `df[mask][col] = value` assigns into a temporary copy that pandas then discards. The data stays as it was. The only sign is a `ChainedAssignmentError` warning. Every regional total still books MF-101 to the North. The "North East" the board was promised does not exist in the numbers.

**The catch.** Treat that warning as an error. After every change, read the result back (`value_counts()`, as above). The correct form is a single `.loc` step, which the cell below shows.

In [ ]:
relabelled.loc[relabelled["store"] == "MF-101", "region"] = "North East"
print(relabelled["region"].value_counts())

### Exhibit C: `groupby` silently dropping a group

**The plausible output.** One store's region is still blank in the master data, because its paperwork is not complete. Asked for "total sales by region, and the company total", the Associate delivers a three-row answer that looks complete. The cell below reproduces it. It also compares the regional sum with the company total.

In [ ]:
audit = daily.copy()
audit.loc[audit["store"] == "MF-112", "region"] = np.nan  # region not yet coded

by_region = audit.groupby("region")["sales"].sum()
print(by_region.round(0))
print(f"\nSum of the regional totals: £{by_region.sum():,.0f}")
print(f"Actual company total:       £{audit['sales'].sum():,.0f}")
print(f"Quietly missing:            £{audit['sales'].sum() - by_region.sum():,.0f}")

**Why it is wrong in business terms.** `groupby()` drops `NaN` keys **by default**. There is no error, no warning, and no `NaN` row. One store's 18 months of takings (about 9% of company sales) are missing from a table headed "total sales by region". A board member who adds up the regional totals and compares them with the annual report finds a gap of about £2.2 million.

**The catch.** Reconcile every time. The parts must sum to the whole, so print both and compare them, as above. When keys can be missing, keep the missing group in the table. `groupby("region", dropna=False)` keeps the `NaN` group, and a board table can label it "unassigned". The cell below shows it.

In [ ]:
print(audit.groupby("region", dropna=False)["sales"].sum().round(0))

### The shared cause

Every exhibit runs without an error. Each failure comes from a silent pandas rule (what `NaN` means, Copy-on-Write, `dropna=True`). An Associate passes it on because the output looks plausible. The defence is a set of checks you run every time:

- Put counts next to means
- Read back after every change
- Check the parts against the whole
- Do one cell by hand

**Next:** Week 6 opens data visualisation design. See the [Week 6 README](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/06/README.md). In Week 7, your team runs its fire-brigade [Sprint Zero](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-sprint-zero), self-paced, and applies this week's work to the group project data.